# Lexsi Track 3 - base model on the hard-30 split (round 3)

The frozen submission adapter (`sft2`) scores **96.67% on hard-30**. One cell of the before/after
table is still empty: Tiny Aya **base**, no adapter, on the *same* 30 tasks.

Round 2 reported easy-SFT (63.33%) as the "before" column, which is not the honest baseline. This
run gives base -> SFT-mix on identical tasks with identical decoding, and re-runs sft2 so
`summary.json` carries the new `overcall` / `parse_failures` fields natively instead of being
patched afterwards.

Nothing is trained here. The adapter is pulled from `primce/tiny-aya-hinglish-lora`, merged back
into the base weights exactly the way round 2 merged it, then decoded greedily.

Expected wall time on T4: ~2 h (base is loopy, up to 8 generations per task). Everything lands in
`runs/` and is uploaded to `primce/lexsi-runs` at the end, so it is safe to leave running.


In [ ]:
import os
from kaggle_secrets import UserSecretsClient

try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN set")
except Exception as exc:
    raise SystemExit(
        "HF_TOKEN is not attached to this notebook. CohereLabs/tiny-aya-global is a gated repo, "
        "so the token is needed to load the model, not just for uploads. "
        "Attach it in Add-ons -> Secrets, then Save & Run All again."
    ) from exc

!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working
rm -rf lexsi-agentic-hinglish
git clone -q https://github.com/PRINCE2-AI/lexsi-agentic-hinglish.git
cd lexsi-agentic-hinglish
git log --oneline -1
python -m pip install -q -r requirements.txt
python data/build_hard.py
python data/build_mix.py
python tests/test_hard_data.py
python tests/test_rewards.py
python tests/test_stats.py
CUDA_VISIBLE_DEVICES=0 python scripts/run_oracle_eval.py --tasks data/tasks_hard_eval.jsonl --out runs/oracle

In [ ]:
%%bash
cd /kaggle/working/lexsi-agentic-hinglish
set -uo pipefail
CUDA_VISIBLE_DEVICES=0 python scripts/run_baseline.py --model CohereLabs/tiny-aya-global \
  --tasks data/tasks_hard_eval.jsonl --no-4bit --out runs/base_hard30 || echo "BASE HARD30 FAILED"

In [ ]:
import torch
from huggingface_hub import snapshot_download
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE = "CohereLabs/tiny-aya-global"
REPO = snapshot_download(
    "primce/tiny-aya-hinglish-lora",
    allow_patterns=["sft2_adapter/*"],
    local_dir="/kaggle/working/sft2_repo",
)
MERGED = "/kaggle/working/sft2_merged"
print("adapter repo:", REPO)

tokenizer = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(model, REPO, subfolder="sft2_adapter")
merged = model.merge_and_unload()
merged.save_pretrained(MERGED, safe_serialization=True)
tokenizer.save_pretrained(MERGED)
del model, merged
torch.cuda.empty_cache()
print("merged ->", MERGED)

In [ ]:
%%bash
cd /kaggle/working/lexsi-agentic-hinglish
set -uo pipefail
if [ ! -f /kaggle/working/sft2_merged/config.json ]; then
  echo "SFT2 MERGE MISSING, skipping tuned eval"
  exit 0
fi
CUDA_VISIBLE_DEVICES=0 python scripts/run_baseline.py --model /kaggle/working/sft2_merged \
  --tasks data/tasks_hard_eval.jsonl --no-4bit --out runs/sft2_hard30_v2 || echo "SFT2 HARD30 FAILED"
CUDA_VISIBLE_DEVICES=0 python scripts/run_baseline.py --model /kaggle/working/sft2_merged \
  --tasks data/tasks_eval.jsonl --no-4bit --out runs/sft2_eval_v2 || echo "SFT2 EVAL24 FAILED"

In [ ]:
import json
import os
from pathlib import Path

from huggingface_hub import HfApi

ROOT = Path("/kaggle/working/lexsi-agentic-hinglish")
os.chdir(ROOT)

RUNS_REPO = "primce/lexsi-runs"
NAMES = ("base_hard30", "sft2_hard30_v2", "sft2_eval_v2")
KEYS = ("success", "tac", "answer_ok", "overcall", "strict_rate", "calls", "steps", "seconds")

for name in NAMES:
    path = ROOT / "runs" / name / "summary.json"
    if path.exists():
        summary = json.loads(path.read_text())
        print(name, {key: summary[key] for key in KEYS if key in summary})
    else:
        print(name, "missing")

token = os.environ.get("HF_TOKEN")
if not token:
    print("no HF_TOKEN, skipping upload")
else:
    api = HfApi(token=token)
    for name in NAMES:
        folder = ROOT / "runs" / name
        if not folder.exists():
            continue
        try:
            api.upload_folder(
                repo_id=RUNS_REPO,
                repo_type="dataset",
                folder_path=str(folder),
                path_in_repo=name,
            )
            print(name, "->", RUNS_REPO)
        except Exception as exc:
            print(name, "upload failed:", exc)